# Week 4 — Signal checks and one transparent baseline

**Lane locked:** Refresh / Content Opportunity Scoring

This notebook checks two pre-decision signals on the March 2026 development anchor, freezes one readable rule, writes a ranked human-review queue, and reviews its top ten skeptically. April supplies outcomes for retrospective checking only; June remains sealed.

## 1. Check two signals before writing the rule

The initial rule idea was: *prioritize visible, aging pages whose March CTR is below what their position band normally earns.* Staleness is behind FlyRank's refresh flags, and CTR-versus-position is behind its CTR-fix logic. Both are tested before either is trusted.

In [1]:
from pathlib import Path
import json
import os

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
output_dir = repo_root / "work" / "outputs"
extension_dir = output_dir / ".duckdb_extensions"
extension_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Store a Hugging Face read token as the HF_TOKEN secret; never paste it into a cell."

con = duckdb.connect()
con.execute(f"SET extension_directory='{extension_dir.as_posix()}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

ROOT = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{ROOT}/dim_content.parquet')"

example_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE gsc_impressions > 0 AND gsc_avg_position >= 1
        ) / NULLIF(SUM(gsc_impressions) FILTER (
            WHERE gsc_impressions > 0 AND gsc_avg_position >= 1
        ), 0) AS avg_position,
        COUNT(*) AS available_days
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS outcome_impressions,
        COUNT(*) AS outcome_available_days
    FROM {FACT_APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
), content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MIN(content_created_date) AS content_created_date
    FROM {DIM_CONTENT}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.ctr,
    m.avg_position,
    GREATEST(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'), 0) AS content_age_days,
    a.outcome_impressions,
    CASE WHEN a.outcome_impressions < 0.80 * m.impressions THEN 1 ELSE 0 END AS future_decline
FROM march AS m
JOIN april AS a USING (client_hash_id, content_hash_id)
LEFT JOIN content AS c USING (client_hash_id, content_hash_id)
WHERE m.impressions >= 100
  AND m.available_days >= 20
  AND a.outcome_available_days >= 20
ORDER BY m.client_hash_id, m.content_hash_id
"""

examples = con.sql(example_query).df()
print(f"Development examples: {len(examples):,}")
print(f"Observed April decline base rate: {examples['future_decline'].mean():.1%}")
print("All candidate inputs are measured by the March 31 decision moment; June remains sealed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Development examples: 88,941
Observed April decline base rate: 51.3%
All candidate inputs are measured by the March 31 decision moment; June remains sealed.


### Signal 1 — staleness behind refresh flags

If age alone were a clean staleness signal, decline rate would rise consistently across older buckets. The table prints `n` so small groups cannot masquerade as strong evidence.

In [2]:
examples["age_bucket"] = pd.cut(
    examples["content_age_days"],
    bins=[-1, 89, 179, 364, np.inf],
    labels=["<90 days", "90–179 days", "180–364 days", "365+ days"],
)
age_buckets = (
    examples.groupby("age_bucket", observed=True)["future_decline"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)
display(age_buckets.style.format({"decline_rate": "{:.1%}"}))
print(f"n total: {age_buckets['n'].sum():,}")
print("Verdict: MIXED")

,age_bucket,n,decline_rate
0,<90 days,25695,44.9%
1,90–179 days,15177,61.0%
2,180–364 days,34925,53.8%
3,365+ days,13144,45.8%


n total: 88,941
Verdict: MIXED


**Verdict: MIXED.** The middle-age buckets decline more often, but the 365+ bucket reverses the pattern. A simple “older means riskier” refresh rule is not supported, so the final rule uses a bounded 90–364 day review band rather than rewarding age forever.

### Signal 2 — CTR versus position behind CTR-fix logic

March CTR is compared with the median CTR of the same visible position band (`1–3`, `4–10`, or `11–20`). This avoids calling a low raw CTR surprising when position already explains it. Source position values below 1 are excluded as invalid, and rows beyond position 20 are excluded because they are outside this visible-opportunity rule.

In [3]:
examples["position_band"] = pd.cut(
    examples["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["1–3", "4–10", "11–20"],
    include_lowest=True,
)
examples["expected_ctr"] = examples.groupby("position_band", observed=True)["ctr"].transform("median")
examples["ctr_vs_position"] = examples["ctr"] / examples["expected_ctr"].replace(0, np.nan)
examples["ctr_gap_bucket"] = pd.cut(
    examples["ctr_vs_position"],
    bins=[-np.inf, 0.5, 1.0, 2.0, np.inf],
    labels=["<0.5× expected", "0.5–1× expected", "1–2× expected", "2×+ expected"],
)
ctr_buckets = (
    examples.dropna(subset=["ctr_gap_bucket"])
    .groupby("ctr_gap_bucket", observed=True)["future_decline"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)
display(ctr_buckets.style.format({"decline_rate": "{:.1%}"}))
print(f"n with a valid CTR-position comparison: {ctr_buckets['n'].sum():,}")
print("Verdict: CONFIRMED")

,ctr_gap_bucket,n,decline_rate
0,<0.5× expected,24526,62.2%
1,0.5–1× expected,9361,58.8%
2,1–2× expected,13449,50.2%
3,2×+ expected,20434,36.5%


n with a valid CTR-position comparison: 67,770
Verdict: CONFIRMED


**Verdict: CONFIRMED.** Decline rate falls as March CTR improves relative to its position-band expectation. This is observational and does not prove that rewriting metadata causes recovery, but it is strong enough to keep as the main transparent ranking signal.

## 2. Encode one rule and write the ranked queue

**The one rule:** among visible pages aged 90–364 days with CTR below their March position-band median, rank review priority as 70% CTR shortfall plus 30% log-scaled exposure. The weights and 100,000-impression exposure cap are fixed by hand, not fitted to April labels.

Every queued row receives exactly one reason code, `aging_visible_low_ctr`, and one human action label, `refresh_or_metadata_review`. The rule recommends review only; it never changes content automatically.

In [4]:
SCORE_INPUTS = [
    "impressions", "ctr", "avg_position", "content_age_days",
    "position_band", "expected_ctr", "ctr_vs_position",
]
assert not any(name.startswith(("future_", "outcome_")) for name in SCORE_INPUTS)

examples["ctr_shortfall"] = (1 - examples["ctr_vs_position"]).clip(lower=0, upper=1)
examples["exposure_component"] = (
    np.log1p(examples["impressions"]) / np.log1p(100_000)
).clip(lower=0, upper=1)

candidate = (
    examples["content_age_days"].between(90, 364)
    & examples["position_band"].notna()
    & examples["ctr_shortfall"].gt(0)
)

queue = examples.loc[candidate].copy()
queue["baseline_score"] = 100 * (
    0.70 * queue["ctr_shortfall"]
    + 0.30 * queue["exposure_component"]
)
queue["reason_code"] = "aging_visible_low_ctr"
queue["action_label"] = "refresh_or_metadata_review"
queue = queue.sort_values(
    ["baseline_score", "impressions", "content_hash_id"],
    ascending=[False, False, True],
    kind="stable",
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

public_columns = [
    "rank", "content_hash_id", "baseline_score", "reason_code", "action_label",
    "impressions", "ctr", "expected_ctr", "avg_position", "content_age_days",
]
csv_path = output_dir / "baseline_action_score.csv"
queue[public_columns].to_csv(csv_path, index=False)

base_rate = float(examples["future_decline"].mean())
p10 = float(queue.head(10)["future_decline"].mean())
p50 = float(queue.head(50)["future_decline"].mean())

# Leakage assertion: changing later outcomes cannot change any already-computed score.
scores_before = queue["baseline_score"].copy()
queue["future_decline"] = 1 - queue["future_decline"]
queue["outcome_impressions"] = 0
assert scores_before.equals(queue["baseline_score"])
queue["future_decline"] = examples.loc[queue.index, "future_decline"] if False else queue["future_decline"]

# Restore review outcomes from the source by pseudonymous key after the leakage test.
outcomes = examples.set_index(["client_hash_id", "content_hash_id"])["future_decline"]
queue["future_decline"] = [outcomes.loc[(a, b)] for a, b in zip(queue["client_hash_id"], queue["content_hash_id"])]

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "development_anchor": "2026-03 predicting 2026-04",
    "eligible_rows": int(len(examples)),
    "queue_rows": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": p10,
    "precision_at_50": p50,
    "signal_verdicts": {"staleness": "MIXED", "ctr_vs_position": "CONFIRMED"},
    "reason_code": "aging_visible_low_ctr",
    "action_label": "refresh_or_metadata_review",
    "sealed_month_used": False,
}
metrics_path = output_dir / "w04_baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

print(f"Ranked queue rows: {len(queue):,}")
print(f"Base rate: {base_rate:.1%}")
print(f"Precision@10: {p10:.1%}; Precision@50: {p50:.1%}")
print(f"CSV written: {csv_path.relative_to(repo_root)}")
print(f"Metrics receipt written: {metrics_path.relative_to(repo_root)}")
print("Reason codes per row: exactly one")
print("Future/outcome inputs used by score: none")
display(queue[public_columns].head(10))

Ranked queue rows: 18,897
Base rate: 51.3%
Precision@10: 90.0%; Precision@50: 80.0%
CSV written: work\outputs\baseline_action_score.csv
Metrics receipt written: work\outputs\w04_baseline_metrics.json
Reason codes per row: exactly one
Future/outcome inputs used by score: none


,rank,content_hash_id,baseline_score,reason_code,action_label,impressions,ctr,expected_ctr,avg_position,content_age_days
0,1,content_9c057b66c30a3abb,98.734768,aging_visible_low_ctr,refresh_or_metadata_review,83834.0,0.000012,0.001036,13.292929,243
1,2,content_0c5606abaaab3178,97.537395,aging_visible_low_ctr,refresh_or_metadata_review,38865.0,0.000000,0.001952,4.804554,278
2,3,content_23a42776a7009b65,96.769958,aging_visible_low_ctr,refresh_or_metadata_review,28950.0,0.000000,0.001952,9.563005,292
3,4,content_713b157e9c77690a,96.378115,aging_visible_low_ctr,refresh_or_metadata_review,24908.0,0.000000,0.001952,3.432381,160
4,5,content_17d994b99d470434,95.798198,aging_visible_low_ctr,refresh_or_metadata_review,19938.0,0.000000,0.001036,10.909118,193
5,6,content_93d76695da196fdf,95.712376,aging_visible_low_ctr,refresh_or_metadata_review,19292.0,0.000000,0.001952,8.243572,278
6,7,content_945d6ff91386c817,95.516630,aging_visible_low_ctr,refresh_or_metadata_review,58278.0,0.000086,0.001952,8.331429,278
7,8,content_37a6fac676c8cebb,95.105058,aging_visible_low_ctr,refresh_or_metadata_review,48049.0,0.000083,0.001952,4.231056,195
8,9,content_fa4cf3aa5ce67bb8,95.046956,aging_visible_low_ctr,refresh_or_metadata_review,25588.0,0.000039,0.001952,4.832812,216
9,10,content_fe8baba849843607,95.024012,aging_visible_low_ctr,refresh_or_metadata_review,14813.0,0.000000,0.001952,3.205956,216


## 3. Top-10 skeptical review

Each line below contains the action, why the page ranked, and what could make the recommendation wrong. The April label is read only after ranking to audit the picks.

In [5]:
top10 = queue.head(10).copy()

def review_line(row):
    why = (
        f"{row.impressions:,.0f} March impressions; CTR {row.ctr:.3%} vs "
        f"{row.expected_ctr:.3%} expected in position band; age {row.content_age_days:.0f} days"
    )
    if row.future_decline == 0:
        wrong = "April did not meet the decline label; the CTR gap may reflect intent, SERP layout, or measurement rather than content"
    else:
        wrong = "the observed April decline may be seasonal or measurement-driven, so a refresh may not cause recovery"
    return f"{row.action_label} — {why} — wrong if {wrong}."

top10["review_line"] = top10.apply(review_line, axis=1)
display(top10[["rank", "content_hash_id", "review_line"]])
print(f"Reviewed rows: {len(top10)}")
assert len(top10) == 10
assert top10["review_line"].str.contains("wrong if", case=False).all()

,rank,content_hash_id,review_line
0,1,content_9c057b66c30a3abb,"refresh_or_metadata_review — 83,834 March impr..."
1,2,content_0c5606abaaab3178,"refresh_or_metadata_review — 38,865 March impr..."
2,3,content_23a42776a7009b65,"refresh_or_metadata_review — 28,950 March impr..."
3,4,content_713b157e9c77690a,"refresh_or_metadata_review — 24,908 March impr..."
4,5,content_17d994b99d470434,"refresh_or_metadata_review — 19,938 March impr..."
5,6,content_93d76695da196fdf,"refresh_or_metadata_review — 19,292 March impr..."
6,7,content_945d6ff91386c817,"refresh_or_metadata_review — 58,278 March impr..."
7,8,content_37a6fac676c8cebb,"refresh_or_metadata_review — 48,049 March impr..."
8,9,content_fa4cf3aa5ce67bb8,"refresh_or_metadata_review — 25,588 March impr..."
9,10,content_fe8baba849843607,"refresh_or_metadata_review — 14,813 March impr..."


Reviewed rows: 10


## 4. Weak picks

A weak pick is a top-ten row that did not meet the observed April decline proxy. It is still a legitimate review candidate, but it shows that the rule is a baseline rather than a decision oracle.

In [6]:
weak_picks = top10.loc[top10["future_decline"].eq(0), [
    "rank", "content_hash_id", "baseline_score", "reason_code", "action_label", "review_line"
]]
print(f"Weak picks in top 10: {len(weak_picks)}")
display(weak_picks)
print("Lesson: high exposure plus a CTR gap can still be a false alarm; seasonality, intent, SERP features, and tracking need human review.")

Weak picks in top 10: 1


,rank,content_hash_id,baseline_score,reason_code,action_label,review_line
2,3,content_23a42776a7009b65,96.769958,aging_visible_low_ctr,refresh_or_metadata_review,"refresh_or_metadata_review — 28,950 March impr..."


Lesson: high exposure plus a CTR gap can still be a false alarm; seasonality, intent, SERP features, and tracking need human review.


## 5. Self-check

- [x] The lane remains Refresh / Content Opportunity Scoring.
- [x] Two signal checks have visible bucket tables and `n`.
- [x] Staleness is linked to refresh flags; CTR-versus-position is linked to CTR-fix logic.
- [x] Each signal has a one-word verdict: MIXED or CONFIRMED.
- [x] One transparent rule produces a score, exactly one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Ten rows are reviewed one line each with “what would make it wrong.”
- [x] Weak picks are named and interpreted.
- [x] No future-window or label-derived input enters the score; June remains sealed.
- [x] The queue is for human review, not automatic content changes.